# Chapter 16 — Building Document Retrieval: The Production Knowledge Pipeline

**Volume 2: Practical Applications — AI for Networking and Security Engineers**

In Chapter 14 you learned what RAG is. In Chapter 15 agents used RAG as memory.
Now we build the **production retrieval layer** — the part that determines whether
the entire system works or gives confidently wrong answers.

> *In Simple Words: The retrieval layer is the nervous system of a RAG application.
> The LLM is the brain. If the nervous system sends wrong signals, even a brilliant
> brain makes wrong decisions. Most RAG failures are nervous system failures.*

---
**What you will build — 5 hands-on demos:**

| # | Demo | Key concept |
|---|------|-------------|
| 1 | Document Ingestion Pipeline | Extract → Chunk → Enrich → Index (4 stages) |
| 2 | Contextual Retrieval (Anthropic technique) | AI-generated context per chunk before embedding |
| 3 | Metadata Filtering + Self-Querying | Filter by device/type/date from natural language |
| 4 | Hybrid Search | Semantic + keyword fusion with Reciprocal Rank Fusion |
| 5 | Re-ranking Pipeline | Retrieve wide → Re-rank precise → Generate grounded |

> **Prerequisite:** `!pip install anthropic scikit-learn numpy` (run install cell first)


In [ ]:
!pip install -q anthropic scikit-learn numpy

---
## Demo 1 — Document Ingestion Pipeline: Extract → Chunk → Enrich → Index

Every production RAG system has an ingestion pipeline with **4 stages**:

```
Raw Docs (configs, PDFs, runbooks)
        │
        ▼  Stage 1: EXTRACT
   Clean Text (structure preserved)
        │
        ▼  Stage 2: CHUNK
   Focused Pieces (512 tokens + 10% overlap)
        │
        ▼  Stage 3: ENRICH
   Contextualized Chunks (chunk + source metadata)
        │
        ▼  Stage 4: INDEX
   Vector Store (searchable by meaning)
```

**The most important rule for network configs:**
Preserve stanza hierarchy. `neighbor 10.1.1.1 remote-as 65002` is meaningless
without the `router bgp 65001` parent. Splitting across chunk boundaries = useless fragments.

> **Networking analogy:** Bad extraction is like `show running-config` returning all lines
> in random order with no indentation. You have the data, but the structure — and meaning — is gone.


In [ ]:
# Demo 1: Complete 4-stage ingestion pipeline (no API needed)
import re
import hashlib
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ── Our "raw documents" — 3 different types ──────────────────────────────────
RAW_DOCS = {
    "core-01-config": {
        "source_type": "network_config",
        "hostname":    "core-01",
        "site":        "datacenter",
        "content": """hostname core-01
!
interface GigabitEthernet0/0
 description Uplink-to-ISP-AS65000
 ip address 203.0.113.1 255.255.255.252
 no shutdown
!
interface GigabitEthernet0/1
 description Peer-link-to-core-02
 ip address 10.1.1.1 255.255.255.252
 no shutdown
!
router bgp 65001
 bgp router-id 10.0.0.1
 neighbor 203.0.113.2 remote-as 65000
 neighbor 203.0.113.2 description ISP-Uplink
 neighbor 203.0.113.2 soft-reconfiguration inbound
 neighbor 10.1.1.2 remote-as 65001
 neighbor 10.1.1.2 description iBGP-to-core-02
 network 10.0.0.0 mask 255.0.0.0
!
router ospf 1
 router-id 10.0.0.1
 network 10.0.0.0 0.255.255.255 area 0
 network 172.16.0.0 0.15.255.255 area 0
!
""",
    },
    "bgp-runbook": {
        "source_type": "runbook",
        "hostname":    "all",
        "site":        "all",
        "content": """BGP Troubleshooting Runbook v3.1

Section 1: Neighbor Not Establishing
When BGP neighbor fails to establish, follow this sequence.
Step 1: Verify TCP port 179 is reachable: telnet <neighbor-ip> 179.
If TCP fails, check ACLs blocking port 179, check routing table for return path.
Step 2: Verify AS numbers match: show bgp neighbors | include remote AS.
A wrong AS number causes the OPEN message to be rejected immediately.
Step 3: Check timer configuration. Hold timer must match on both sides.
show bgp neighbors | include hold. Default is 90s hold, 30s keepalive.

Section 2: Routes Not Received
After session establishes, verify route advertisements.
Run: show bgp ipv4 unicast neighbors <ip> received-routes.
If missing routes, check inbound route-map and prefix-list policies.
Also check maximum-prefix limits — hitting the limit suppresses all routes.

Section 3: Session Flapping
BGP flapping is usually a physical or timer problem.
Check interface errors: show interface <int> | include error.
Check BGP hold timer — asymmetric paths with varying RTT can cause timer expiry.
Use BFD for sub-second failure detection instead of relying on BGP timers.
""",
    },
    "incident-2024-001": {
        "source_type": "incident",
        "hostname":    "core-01",
        "site":        "datacenter",
        "content": """Incident Report INC-2024-001
Date: 2024-03-15 02:14 UTC
Severity: P1
Affected: BGP peering with ISP (AS 65000)

Timeline:
02:14 — NOC alerted: BGP session to 203.0.113.2 down
02:18 — Engineer SSH to core-01. show bgp summary: neighbor IDLE
02:22 — Ping to 203.0.113.2 fails. Physical layer issue suspected.
02:25 — show interface GigabitEthernet0/0: input errors 14,823, CRC 14,812
02:27 — Physical inspection: SFP transceiver showing Rx power -22 dBm (min -20)
02:35 — SFP replaced. Interface restored. BGP reconverged in 45 seconds.

Root Cause: Failed SFP transceiver on GigabitEthernet0/0.
Fix: Replace SFP. No config change required.
Prevention: Add optical power monitoring threshold alerts.
""",
    },
}

# ─────────────────────────────────────────────────────────────────────────────
# Stage 1: EXTRACT — parse structure from raw text
# ─────────────────────────────────────────────────────────────────────────────
def extract(doc_id: str, doc: dict) -> list:
    """Extract logical sections preserving hierarchy."""
    content = doc["content"]
    source_type = doc["source_type"]
    sections = []

    if source_type == "network_config":
        # Split at stanza boundaries (lines starting with a keyword or !)
        current = []
        for line in content.splitlines():
            stripped = line.strip()
            if stripped == "!" and current:
                block = "\n".join(current).strip()
                if len(block) > 10:
                    sections.append(block)
                current = []
            else:
                current.append(line)
        if current:
            sections.append("\n".join(current).strip())

    else:  # runbook, incident — split at blank lines or section headers
        paras = re.split(r"\n\n+", content)
        sections = [p.strip() for p in paras if len(p.strip()) > 20]

    return [{"text": s, "doc_id": doc_id, **{k: v for k, v in doc.items() if k != "content"}}
            for s in sections]

# ─────────────────────────────────────────────────────────────────────────────
# Stage 2: CHUNK — split large sections with overlap
# ─────────────────────────────────────────────────────────────────────────────
def chunk_section(section: dict, max_words: int = 80, overlap: int = 15) -> list:
    """Sliding window over words with overlap to prevent boundary splits."""
    words = section["text"].split()
    if len(words) <= max_words:
        return [section]  # Small enough — keep as-is

    chunks = []
    step = max_words - overlap
    for i in range(0, len(words), step):
        window = words[i : i + max_words]
        chunk = dict(section)
        chunk["text"] = " ".join(window)
        chunk["chunk_idx"] = i // step
        chunks.append(chunk)
    return chunks

# ─────────────────────────────────────────────────────────────────────────────
# Stage 3: ENRICH — add metadata fingerprint and header context
# ─────────────────────────────────────────────────────────────────────────────
def enrich(chunk: dict) -> dict:
    """Add document header context + unique fingerprint."""
    enriched = dict(chunk)
    # Prepend lightweight context (no API call needed — just header info)
    header = (
        f"[Document: {chunk['doc_id']} | "
        f"Type: {chunk['source_type']} | "
        f"Host: {chunk.get('hostname', 'unknown')} | "
        f"Site: {chunk.get('site', 'unknown')}]"
    )
    enriched["text_for_embedding"] = f"{header}\n{chunk['text']}"
    enriched["chunk_id"] = hashlib.md5(enriched["text_for_embedding"].encode()).hexdigest()[:8]
    return enriched

# ─────────────────────────────────────────────────────────────────────────────
# Stage 4: INDEX — build TF-IDF vector store
# ─────────────────────────────────────────────────────────────────────────────
class VectorStore:
    """Simple in-memory vector store backed by TF-IDF."""

    def __init__(self):
        self.chunks     = []
        self.vectorizer = None
        self.matrix     = None

    def add(self, chunks: list):
        self.chunks.extend(chunks)

    def build_index(self):
        texts = [c["text_for_embedding"] for c in self.chunks]
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
        self.matrix     = self.vectorizer.fit_transform(texts)
        return self

    def search(self, query: str, top_k: int = 3, filters: dict = None) -> list:
        q_vec  = self.vectorizer.transform([query])
        scores = cosine_similarity(q_vec, self.matrix).flatten()

        # Apply metadata filters
        results = []
        for i, (score, chunk) in enumerate(zip(scores, self.chunks)):
            if filters:
                match = all(chunk.get(k) == v for k, v in filters.items())
                if not match:
                    continue
            results.append((score, chunk))

        results.sort(key=lambda x: x[0], reverse=True)
        return results[:top_k]


# ─── Run the full pipeline ────────────────────────────────────────────────────
store = VectorStore()
total_chunks = 0

print("="*60)
print("  INGESTION PIPELINE REPORT")
print("="*60)

for doc_id, doc in RAW_DOCS.items():
    # Stage 1: Extract
    sections = extract(doc_id, doc)
    # Stage 2: Chunk
    all_chunks = []
    for sec in sections:
        all_chunks.extend(chunk_section(sec, max_words=80))
    # Stage 3: Enrich
    enriched = [enrich(c) for c in all_chunks]
    # Stage 4: Index
    store.add(enriched)
    total_chunks += len(enriched)
    print(f"\n  [{doc_id}]")
    print(f"    Sections extracted : {len(sections)}")
    print(f"    Chunks produced    : {len(enriched)}")
    print(f"    Sample chunk ID    : {enriched[0]['chunk_id']}")

store.build_index()
print(f"\n  Total chunks indexed : {total_chunks}")
print(f"  Vocabulary size      : {len(store.vectorizer.vocabulary_):,} terms")

# ─── Test the index ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  SEARCH TEST")
print("="*60)
query = "BGP neighbor not establishing AS number mismatch"
results = store.search(query, top_k=3)
print(f'\n  Query: "{query}"')
for score, chunk in results:
    print(f"\n  [{score:.3f}] {chunk['doc_id']} | {chunk['source_type']}")
    print(f"    {chunk['text'][:120]}...")


---
## Demo 2 — Contextual Retrieval: The Single Biggest Quality Improvement

**The problem:** A chunk extracted from a document loses its position in the hierarchy.

`"the default hello interval is 10 seconds"` — is this OSPF? EIGRP? IS-IS? BGP keepalive?
Without context, the embedding is ambiguous. Retrieval for "OSPF hello timer" may miss it.

**Anthropic's solution — Contextual Retrieval:**
Before embedding each chunk, ask a small LLM (Haiku) to write a 1-2 sentence context
that situates this chunk within its parent document.

```
Original chunk:  "the default hello interval is 10 seconds..."
After context:   "[OSPF Design, Datacenter Network v2.4, Section: Timer Config]
                  This chunk describes OSPF hello interval defaults on broadcast networks.
                  the default hello interval is 10 seconds..."
```

Now the embedding captures OSPF + hello + timer → retrieved correctly every time.

**Cost:** One Haiku call per chunk at ingestion time. For 1000 chunks ≈ $0.05 total.
A one-time cost with **permanent** improvement in retrieval quality.

> **Networking analogy:** Like writing on every sticky note: "From John's BGP runbook,
> written during the 2024 DC migration." Now every note is findable by context,
> not just by its own words.


In [ ]:
# Demo 2: Contextual Retrieval — AI-generated context per chunk
import os
from anthropic import Anthropic
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')
client = Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"

# ── Our network documentation chunks ─────────────────────────────────────────
PARENT_DOC = """Network Design Document: Datacenter Core v2.4
Last updated: 2024-01-15

Section 1: OSPF Design
Area 0 is the backbone connecting all distribution blocks.
Hello interval: 10 seconds on broadcast links, 30 seconds on NBMA.
Dead interval: 4x hello interval (40s broadcast, 120s NBMA).
Authentication: MD5 on all OSPF interfaces in area 0.

Section 2: BGP Design
iBGP mesh between core-01 and core-02 for redundancy.
eBGP peering with ISP (AS 65000) on core-01 only.
Hold timer: 90 seconds. Keepalive: 30 seconds.
BGP communities used for traffic engineering: 65001:100 (prefer), 65001:200 (backup).

Section 3: QoS Policy
DSCP markings preserved across the core.
EF (46) for voice, AF41 (34) for video, CS3 (24) for signaling, BE (0) for data.
Policing at edge, scheduling at core. No marking or re-marking in core.
"""

CHUNKS = [
    "the default hello interval is 10 seconds on broadcast links, 30 seconds on NBMA.",
    "BGP hold timer is 90 seconds. Keepalive is 30 seconds.",
    "Authentication: MD5 on all OSPF interfaces in area 0.",
    "BGP communities used for traffic engineering: 65001:100 prefer, 65001:200 backup.",
    "DSCP markings preserved across the core. EF (46) for voice, AF41 (34) for video.",
]

# ── Generate context for a chunk ─────────────────────────────────────────────
def generate_context(parent_document: str, chunk: str) -> str:
    """Ask Haiku to situate this chunk within the parent document."""
    prompt = (
        f"Here is a network design document:\n"
        f"<document>{parent_document[:2000]}</document>\n\n"
        f"Here is a specific chunk from this document:\n"
        f"<chunk>{chunk}</chunk>\n\n"
        f"Write 1-2 sentences that explain what this chunk is about "
        f"and which section/topic it belongs to. Be specific about the protocol "
        f"and parameter. Do not repeat the chunk text."
    )
    r = client.messages.create(
        model=HAIKU,
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text.strip()

# ── Build two indexes: plain and contextual ───────────────────────────────────
print("Building contextual chunk indexes...\n")

plain_texts       = []
contextual_texts  = []
contexts_generated = []

for i, chunk in enumerate(CHUNKS):
    # Plain: just the chunk text
    plain_texts.append(chunk)

    # Contextual: AI context + chunk text
    ctx = generate_context(PARENT_DOC, chunk)
    contexts_generated.append(ctx)
    contextual_texts.append(f"{ctx}\n{chunk}")
    print(f"  Chunk {i+1}: {chunk[:60]}...")
    print(f"  Context: {ctx}\n")

# Build TF-IDF for both
vec_plain = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
mat_plain = vec_plain.fit_transform(plain_texts)

vec_ctx   = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
mat_ctx   = vec_ctx.fit_transform(contextual_texts)

# ── Compare retrieval quality ─────────────────────────────────────────────────
def search_plain(query: str, top_k: int = 2) -> list:
    q = vec_plain.transform([query])
    scores = cosine_similarity(q, mat_plain).flatten()
    top = np.argsort(-scores)[:top_k]
    return [(scores[i], CHUNKS[i]) for i in top]

def search_contextual(query: str, top_k: int = 2) -> list:
    q = vec_ctx.transform([query])
    scores = cosine_similarity(q, mat_ctx).flatten()
    top = np.argsort(-scores)[:top_k]
    return [(scores[i], CHUNKS[i]) for i in top]

print("\n" + "="*60)
print("  RETRIEVAL QUALITY COMPARISON")
print("="*60)

test_queries = [
    "OSPF hello timer configuration",          # chunk 0 — about OSPF
    "BGP keepalive interval",                   # chunk 1 — about BGP timers
    "traffic engineering BGP communities",      # chunk 3 — about BGP communities
]

for q in test_queries:
    print(f'\n  Query: "{q}"')

    plain_res = search_plain(q, top_k=1)
    ctx_res   = search_contextual(q, top_k=1)

    print(f"  Plain   [{plain_res[0][0]:.3f}]: {plain_res[0][1][:70]}")
    print(f"  Context [{ctx_res[0][0]:.3f}]:  {ctx_res[0][1][:70]}")

    improvement = ctx_res[0][0] - plain_res[0][0]
    icon = "✅" if improvement >= 0 else "⚠️"
    print(f"  {icon} Similarity change: {improvement:+.3f}")


---
## Demo 3 — Metadata Filtering + Self-Querying Retrieval

**The problem:** Sometimes you don't want to search by meaning alone.
You want to filter first, then search within that subset.

Examples:
- "Search only `core-01` configs, not the runbooks"
- "Search only incident reports from datacenter site"
- "Search only BGP-related documentation"

**The solution:** Attach structured metadata to every chunk at ingestion time.
Use metadata as a **pre-filter** before the similarity search.

> **Networking analogy:** Metadata filtering is like a VRF route lookup.
> You don't search the global routing table — you first select the VRF (filter),
> then look up the route within that VRF (similarity search within filtered set).

**Self-Querying:** The user just asks a natural language question.
Claude extracts both the semantic query AND the metadata filter automatically.

```
User: "What BGP timers are configured on core-01?"
Claude extracts:
  → semantic query: "BGP timer configuration keepalive hold"
  → filter: {"source_type": "network_config", "hostname": "core-01"}
```


In [ ]:
from anthropic import Anthropic
# Demo 3: Metadata filtering + self-querying retrieval
import numpy as np
import json
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

client = Anthropic()
HAIKU = "claude-haiku-4-5-20251001"

# ── Knowledge base with rich metadata ────────────────────────────────────────
KB = [
    {
        "text": "router bgp 65001\n neighbor 203.0.113.2 remote-as 65000\n neighbor 203.0.113.2 description ISP\n bgp timers 30 90",
        "source_type": "network_config", "hostname": "core-01", "site": "datacenter", "protocol": "bgp",
    },
    {
        "text": "router bgp 65001\n neighbor 10.1.1.1 remote-as 65001\n neighbor 10.1.1.1 description iBGP-core-01\n bgp timers 30 90",
        "source_type": "network_config", "hostname": "core-02", "site": "datacenter", "protocol": "bgp",
    },
    {
        "text": "router ospf 1\n router-id 10.0.0.1\n area 0 authentication message-digest\n timers throttle spf 50 200 5000",
        "source_type": "network_config", "hostname": "core-01", "site": "datacenter", "protocol": "ospf",
    },
    {
        "text": "BGP hold timer must match on both peers. Default 90s hold, 30s keepalive. Mismatch causes Hold Timer Expired notifications.",
        "source_type": "runbook", "hostname": "all", "site": "all", "protocol": "bgp",
    },
    {
        "text": "OSPF hello/dead intervals must match on same segment. Default: hello 10s, dead 40s broadcast. Mismatch keeps neighbor in INIT state.",
        "source_type": "runbook", "hostname": "all", "site": "all", "protocol": "ospf",
    },
    {
        "text": "INC-2024-001: BGP to ISP down. Root cause: Failed SFP on GigabitEthernet0/0. CRC errors 14,812. Fix: SFP replacement.",
        "source_type": "incident", "hostname": "core-01", "site": "datacenter", "protocol": "bgp",
    },
    {
        "text": "INC-2024-007: OSPF adjacency flapping between core-01 and dist-01. Root cause: MTU mismatch. Fix: ip ospf mtu-ignore on dist-01.",
        "source_type": "incident", "hostname": "dist-01", "site": "datacenter", "protocol": "ospf",
    },
    {
        "text": "router bgp 65001\n neighbor 10.2.2.2 remote-as 65001\n neighbor 10.2.2.2 description Branch-VPN-hub\n bgp timers 10 30",
        "source_type": "network_config", "hostname": "branch-01", "site": "branch", "protocol": "bgp",
    },
]

# Build the index
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
matrix = vectorizer.fit_transform([item["text"] for item in KB])

# ── Filtered search ────────────────────────────────────────────────────────────
def filtered_search(query: str, filters: dict = None, top_k: int = 3) -> list:
    """Semantic search with optional metadata pre-filtering."""
    q_vec  = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, matrix).flatten()

    results = []
    for i, (score, item) in enumerate(zip(scores, KB)):
        if filters:
            match = all(item.get(k) == v for k, v in filters.items())
            if not match:
                continue
        results.append((score, item))

    results.sort(key=lambda x: x[0], reverse=True)
    return results[:top_k]

# ── Self-querying: LLM extracts filter from natural language ──────────────────
FILTER_SCHEMA = {
    "source_type": ["network_config", "runbook", "incident"],
    "hostname": ["core-01", "core-02", "branch-01", "dist-01", "all"],
    "site": ["datacenter", "branch", "all"],
    "protocol": ["bgp", "ospf"],
}

def self_query(natural_question: str) -> dict:
    """Use Claude to extract semantic query + metadata filter from a question."""
    prompt = (
        f"Parse this network question and extract a semantic search query and metadata filters.\n\n"
        f"Question: {natural_question}\n\n"
        f"Available filter fields and values:\n{json.dumps(FILTER_SCHEMA, indent=2)}\n\n"
        f"Respond ONLY with valid JSON in this exact format:\n"
        f'{{"semantic_query": "...", "filters": {{"field": "value"}} }}\n'
        f"Only include filters you are confident about. Use empty dict if no filter applies."
    )
    resp = client.messages.create(
        model=HAIKU,
        max_tokens=150,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = resp.content[0].text.strip()
    try:
        json_match = re.search(r'\{[\s\S]*\}', raw)
        return json.loads(json_match.group()) if json_match else {"semantic_query": natural_question, "filters": {}}
    except Exception:
        return {"semantic_query": natural_question, "filters": {}}

# ── Run demos ──────────────────────────────────────────────────────────────────
print("="*60)
print("  SELF-QUERYING RETRIEVAL DEMO")
print("="*60)

questions = [
    "What BGP timers are configured on core-01?",
    "Show me all BGP incident reports from the datacenter",
    "What OSPF timer defaults should I use?",
]

for q in questions:
    print(f'\n  Question: "{q}"')

    # Self-query parsing
    parsed = self_query(q)
    sem_q   = parsed.get("semantic_query", q)
    filters = parsed.get("filters", {})

    print(f'  Extracted semantic query: "{sem_q}"')
    print(f"  Extracted filters: {filters}")

    # Search with filters
    results = filtered_search(sem_q, filters=filters if filters else None, top_k=2)
    print(f"  Results ({len(results)} returned):")
    for score, item in results:
        tag = f"{item['source_type']}:{item['hostname']}"
        print(f"    [{score:.3f}] [{tag}] {item['text'][:80]}...")


---
## Demo 4 — Hybrid Search: Semantic + Keyword Fusion

**The problem with pure semantic search:**
If a user searches for a specific device name like `GigabitEthernet0/1`
or a specific error code like `INC-2024-001`, semantic search may miss it —
these terms are so specific they don't exist in the embedding vocabulary.

**The problem with pure keyword search:**
"BGP session is broken" won't find "neighbor not establishing"
because there's no word overlap.

**The solution: Hybrid Search**
Run BOTH types of search, then merge results using **Reciprocal Rank Fusion (RRF)**.

```
Semantic search results:  [doc3=0.82, doc1=0.71, doc5=0.65]
Keyword search results:   [doc1=0.91, doc6=0.78, doc3=0.55]
                                  │
                          RRF merges them
                                  │
Final ranked list:        [doc1, doc3, doc6, doc5]
```

**RRF formula:** `score(d) = Σ 1 / (k + rank(d))` where k=60 (constant)
- Documents appearing high in multiple lists get the highest combined scores
- No need to normalize scores from different systems — only ranks matter

> **Networking analogy:** Hybrid search is like Equal-Cost Multi-Path (ECMP).
> Multiple paths (search methods) reach the same destination (relevant documents).
> RRF is the load balancing: results that appear via multiple paths get priority.


In [ ]:
# Demo 4: Hybrid search with Reciprocal Rank Fusion (RRF)
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import math

# ── Knowledge base ────────────────────────────────────────────────────────────
DOCS = [
    {"id": "D1", "text": "BGP neighbor 10.1.1.2 not establishing. Check AS numbers with show bgp neighbors."},
    {"id": "D2", "text": "Interface GigabitEthernet0/1 showing CRC errors 14823. Likely SFP failure or duplex mismatch."},
    {"id": "D3", "text": "OSPF adjacency stuck in EXSTART. MTU mismatch between peers. Use ip ospf mtu-ignore."},
    {"id": "D4", "text": "BGP hold timer expired on peer 10.1.1.2. Local timer 90s, remote timer 180s. Mismatch."},
    {"id": "D5", "text": "NTP stratum 2 server unreachable. Check routing table and firewall rules for UDP 123."},
    {"id": "D6", "text": "INC-2024-001: BGP peer 203.0.113.2 session down. SFP failure on GigabitEthernet0/0."},
    {"id": "D7", "text": "BGP route 10.0.0.0/8 not received from peer. Check inbound route-map prefix-list filters."},
    {"id": "D8", "text": "OSPF neighbor in INIT state. Hello packets not reaching peer. Check ACL and multicast."},
]

texts = [d["text"] for d in DOCS]

# ── Build TF-IDF semantic index ───────────────────────────────────────────────
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
mat = vec.fit_transform(texts)

def semantic_search(query: str, top_k: int = 5) -> list:
    """Returns list of (doc_id, score) sorted by score."""
    q_vec  = vec.transform([query])
    scores = cosine_similarity(q_vec, mat).flatten()
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [(DOCS[i]["id"], score) for i, score in ranked]

# ── Simple keyword search (BM25-lite: term overlap + IDF weighting) ───────────
import math

def idf(term: str, docs: list) -> float:
    """Inverse document frequency — rare terms score higher."""
    n_containing = sum(1 for d in docs if term.lower() in d.lower())
    if n_containing == 0:
        return 0.0
    return math.log((len(docs) + 1) / (n_containing + 0.5))

def keyword_search(query: str, top_k: int = 5) -> list:
    """Term-overlap keyword search with IDF weighting."""
    query_terms = [t.lower() for t in query.split() if len(t) > 2]
    scores = []
    for doc in DOCS:
        doc_lower = doc["text"].lower()
        score = sum(
            idf(term, texts) * doc_lower.count(term)
            for term in query_terms
        )
        scores.append((doc["id"], score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

# ── Reciprocal Rank Fusion ────────────────────────────────────────────────────
def reciprocal_rank_fusion(ranked_lists: list, k: int = 60) -> list:
    """
    Merge multiple ranked lists using RRF.
    score(d) = sum of 1/(k + rank(d)) across all lists.
    Higher score = document appeared highly ranked in more lists.
    """
    rrf_scores = {}
    for ranked_list in ranked_lists:
        for rank, (doc_id, _) in enumerate(ranked_list, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1.0 / (k + rank)

    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

def hybrid_search(query: str, top_k: int = 3) -> list:
    """Hybrid: semantic + keyword, merged with RRF."""
    sem = semantic_search(query, top_k=5)
    kw  = keyword_search(query, top_k=5)
    fused = reciprocal_rank_fusion([sem, kw])
    return fused[:top_k]

# ── Compare all three methods ─────────────────────────────────────────────────
print("="*65)
print("  HYBRID SEARCH vs INDIVIDUAL METHODS")
print("="*65)

test_cases = [
    ("BGP session down neighbor not establishing", "Semantic should win — paraphrase match"),
    ("INC-2024-001 SFP GigabitEthernet0/0",        "Keyword should win — exact IDs"),
    ("BGP peer hold timer mismatch INC-2024",       "Hybrid wins — needs both"),
]

for query, note in test_cases:
    print(f'\n  Query: "{query}"')
    print(f"  Note:  {note}")

    sem = semantic_search(query, top_k=3)
    kw  = keyword_search(query, top_k=3)
    hyb = hybrid_search(query, top_k=3)

    print(f"\n  Semantic:  {[d for d, _ in sem]}")
    print(f"  Keyword:   {[d for d, _ in kw]}")
    print(f"  Hybrid:    {[d for d, _ in hyb]}  ← merged best of both")

    # Show top result text
    top_id = hyb[0][0]
    top_doc = next(d for d in DOCS if d["id"] == top_id)
    print(f"\n  Top result [{top_id}]: {top_doc['text'][:90]}...")


---
## Demo 5 — Full Production Pipeline: Retrieve Wide, Re-rank Precise, Generate Grounded

This is the complete production retrieval pipeline used in real RAG systems:

```
User Query
    │
    ▼ Stage 1: HYBRID SEARCH (wide net — top 10)
Candidate Chunks (semantic + keyword fusion)
    │
    ▼ Stage 2: RE-RANKING (Claude scores relevance precisely)
Top 3 Highly Relevant Chunks
    │
    ▼ Stage 3: CONTEXTUAL COMPRESSION (extract only the relevant part)
Compressed Context (dense, no noise)
    │
    ▼ Stage 4: AUGMENTED GENERATION (Claude Sonnet answers grounded)
Final Answer + Citations
```

**Why two retrieval stages?**
- Stage 1 (wide): Fast approximate search — maximizes **recall**
- Stage 2 (narrow): Expensive precise re-ranking — maximizes **precision**
- You can't run the expensive re-ranker on the full database — only on candidates

> **Networking analogy:** Two-stage architecture = fast forwarding path + slow path.
> Most packets take the fast ASIC path. Complex decisions go to the CPU slow path.
> Same principle: most retrieval goes through ANN fast path, top-K re-ranking on CPU.


In [ ]:
from anthropic import Anthropic
# Demo 5: Full production pipeline — hybrid retrieve + LLM re-rank + generate
import numpy as np
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import math

client = Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

# ── Rich network knowledge base ──────────────────────────────────────────────
KB = [
    {"id": "KB01", "text": "BGP neighbor 10.1.1.2 state IDLE: verify TCP port 179 reachable with telnet 10.1.1.2 179.", "meta": {"protocol": "bgp"}},
    {"id": "KB02", "text": "BGP hold timer mismatch: local 90s, remote 180s. Fix: neighbor 10.1.1.2 timers 60 180.", "meta": {"protocol": "bgp"}},
    {"id": "KB03", "text": "BGP AS number mismatch causes OPEN message rejection. Verify: show bgp neighbors | include remote AS.", "meta": {"protocol": "bgp"}},
    {"id": "KB04", "text": "BGP MD5 authentication mismatch causes TCP RST. Both sides need same password under neighbor.", "meta": {"protocol": "bgp"}},
    {"id": "KB05", "text": "BGP route not received: check inbound route-map with show bgp neighbors <ip> received-routes.", "meta": {"protocol": "bgp"}},
    {"id": "KB06", "text": "OSPF EXSTART state: MTU mismatch between peers. Fix: ip ospf mtu-ignore on both interfaces.", "meta": {"protocol": "ospf"}},
    {"id": "KB07", "text": "OSPF INIT state: hello packets one-directional. Check multicast 224.0.0.5 ACL and interface passive-interface.", "meta": {"protocol": "ospf"}},
    {"id": "KB08", "text": "OSPF hello/dead timer mismatch: both sides must match. Default: hello 10s, dead 40s on broadcast.", "meta": {"protocol": "ospf"}},
    {"id": "KB09", "text": "Interface CRC errors indicate physical problem: bad cable, SFP, or duplex mismatch. Check show interface.", "meta": {"protocol": "interface"}},
    {"id": "KB10", "text": "Duplex mismatch: one side full, other half — causes late collisions. Fix: duplex full on both.", "meta": {"protocol": "interface"}},
    {"id": "KB11", "text": "NTP synchronization required for log correlation. Configure: ntp server <ip> prefer. Verify: show ntp status.", "meta": {"protocol": "ntp"}},
    {"id": "KB12", "text": "BGP maximum-prefix limit: if hit, session drops with prefix limit exceeded notification. Increase or reset.", "meta": {"protocol": "bgp"}},
]
texts = [k["text"] for k in KB]

# ── Build index ───────────────────────────────────────────────────────────────
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
mat = vec.fit_transform(texts)

def idf(term, docs):
    n = sum(1 for d in docs if term.lower() in d.lower())
    return math.log((len(docs)+1)/(n+0.5)) if n else 0.0

def semantic_search(query, top_k=6):
    q = vec.transform([query])
    scores = cosine_similarity(q, mat).flatten()
    return sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]

def keyword_search(query, top_k=6):
    terms = [t.lower() for t in query.split() if len(t) > 2]
    scored = []
    for i, doc_text in enumerate(texts):
        score = sum(idf(t, texts) * doc_text.lower().count(t) for t in terms)
        scored.append((i, score))
    return sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]

def rrf_merge(lists, k=60):
    scores = {}
    for lst in lists:
        for rank, (idx, _) in enumerate(lst, 1):
            scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

# ── Stage 2: LLM-based re-ranker ─────────────────────────────────────────────
def llm_rerank(query: str, candidates: list, top_k: int = 3) -> list:
    """Use Haiku to score each candidate's relevance to the query (0-10)."""
    candidate_texts = "\n".join(
        f"[{i+1}] {KB[idx]['text']}" for i, (idx, _) in enumerate(candidates)
    )
    prompt = (
        f"Score each document's relevance to the query on a scale of 0-10.\n\n"
        f"Query: {query}\n\n"
        f"Documents:\n{candidate_texts}\n\n"
        f'Respond ONLY with JSON: {{"scores": [score1, score2, ...]}}\n'
        f"One score per document in the same order."
    )
    resp = client.messages.create(
        model=HAIKU,
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}],
    )
    import re
    try:
        raw = resp.content[0].text.strip()
        match = re.search(r'\[[\d\s,\.]+\]', raw)
        scores = json.loads(match.group()) if match else [5] * len(candidates)
    except Exception:
        scores = [5] * len(candidates)

    # Combine index with re-rank scores
    scored = [(candidates[i][0], scores[i]) for i in range(min(len(candidates), len(scores)))]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

# ── Stage 4: RAG Generation ───────────────────────────────────────────────────
def generate_answer(question: str, top_chunks: list) -> str:
    """Generate answer grounded in retrieved context."""
    context = "\n\n".join(
        f"[Source {i+1}] {KB[idx]['text']}"
        for i, (idx, _) in enumerate(top_chunks)
    )
    system = (
        "You are a network operations expert. Answer the engineer's question using "
        "ONLY the provided documentation. Cite which source you used. "
        "If the context doesn't contain the answer, say so explicitly."
    )
    resp = client.messages.create(
        model=SONNET,
        max_tokens=300,
        system=system,
        messages=[{"role": "user", "content": f"Documentation:\n{context}\n\nQuestion: {question}"}],
    )
    return resp.content[0].text.strip()

# ── Run the full pipeline ─────────────────────────────────────────────────────
def full_rag_pipeline(question: str):
    print(f"\n{'='*65}")
    print(f"  QUESTION: {question}")
    print(f"{'='*65}")

    # Stage 1: Hybrid search (wide net)
    sem = semantic_search(question, top_k=6)
    kw  = keyword_search(question, top_k=6)
    fused = rrf_merge([sem, kw])[:8]
    print(f"\n  Stage 1 — Hybrid search: {len(fused)} candidates")
    for idx, score in fused[:4]:
        print(f"    RRF={score:.4f}  {KB[idx]['text'][:65]}...")

    # Stage 2: Re-rank (precise)
    reranked = llm_rerank(question, fused, top_k=3)
    print(f"\n  Stage 2 — Re-ranked top 3:")
    for idx, score in reranked:
        print(f"    Relevance={score}/10  {KB[idx]['text'][:65]}...")

    # Stage 4: Generate
    print(f"\n  Stage 4 — Generated Answer:")
    answer = generate_answer(question, reranked)
    print(f"  {answer}")
    return answer

# Run on two questions
full_rag_pipeline("BGP neighbor is stuck in IDLE state — what should I check?")
full_rag_pipeline("OSPF adjacency won't reach FULL state — stays in EXSTART")


---
## Summary — What You Built

A complete production document retrieval pipeline, from raw text to grounded answers:

### Demo progression:
1. **Ingestion Pipeline** — 4 stages: Extract (preserve stanza hierarchy) → Chunk (overlap) → Enrich (metadata header) → Index (TF-IDF vector store)
2. **Contextual Retrieval** — Haiku generates 1-2 sentence context per chunk at index time → retrieval precision improves significantly
3. **Metadata Filtering + Self-Querying** — Claude extracts device/type/protocol filters from natural language → VRF-style scoped search
4. **Hybrid Search + RRF** — TF-IDF semantic + IDF-weighted keyword, fused with Reciprocal Rank Fusion → best of both worlds
5. **Full Production Pipeline** — Hybrid retrieve (wide) → LLM re-rank (precise) → Sonnet generate (grounded + cited)

### The retrieval quality equation:
```
answer_quality = f(retrieval_recall × retrieval_precision × context_richness)
```

Prioritize in this order:
1. **Recall first** — use hybrid search to not miss relevant chunks
2. **Precision second** — use re-ranking to eliminate noise before generation
3. **Context last** — use contextual retrieval to make every chunk self-explanatory

### Chunking rules for network configs:
- **Never split a stanza** — `router bgp 65001` must stay with its neighbor statements
- **Use 10-20% overlap** — prevents answers from falling through boundary cracks
- **Always prepend context** — source type + hostname + section tells the embedding what the chunk is about

---
*Chapter 16 — Building Document Retrieval | AI for Networking and Security Engineers*
